In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_excel("/content/drive/MyDrive/Fashion_Sales_Cleaned.xlsx", sheet_name="Sheet1")

# Drop unnecessary columns
df.drop(columns=["Index", "Order ID", "Cust ID", "Date"], inplace=True)

# Define features and target
X = df.drop(columns=["Amount"])
y = df["Amount"]

# Standardize numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Model training and evaluation
def train_and_evaluate(model, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    accuracy = r2 * 100
    print(f"{name}: RMSE = {rmse:.4f}, R^2 = {r2:.4f}, Accuracy = {accuracy:.2f}%")

# Linear Regression
train_and_evaluate(LinearRegression(), "Linear Regression")

# Decision Tree with tuning
dt = DecisionTreeRegressor()
dt_params = {"max_depth": [5, 10, 20, None]}
dt_grid = GridSearchCV(dt, dt_params, cv=5, n_jobs=-1)
dt_grid.fit(X_train, y_train)
train_and_evaluate(dt_grid.best_estimator_, "Decision Tree")

# Random Forest with tuning
rf = RandomForestRegressor(n_jobs=-1, random_state=42)
rf_params = {"n_estimators": [50, 100, 200], "max_depth": [10, 20, None]}
rf_grid = GridSearchCV(rf, rf_params, cv=3, n_jobs=-1)
rf_grid.fit(X_train, y_train)
train_and_evaluate(rf_grid.best_estimator_, "Random Forest")

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
train_and_evaluate(gb, "Gradient Boosting")

# XGBoost
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, n_jobs=-1, random_state=42)
train_and_evaluate(xgb, "XGBoost")


Linear Regression: RMSE = 0.8613, R^2 = 0.2769, Accuracy = 27.69%
Decision Tree: RMSE = 0.4405, R^2 = 0.8108, Accuracy = 81.08%
Random Forest: RMSE = 0.3758, R^2 = 0.8623, Accuracy = 86.23%
Gradient Boosting: RMSE = 0.4107, R^2 = 0.8356, Accuracy = 83.56%
XGBoost: RMSE = 0.4966, R^2 = 0.7597, Accuracy = 75.97%
